# 06 - Can we fix neutral? (E9)

In notebook 04 every transformer got **0 F1 on neutral**: they never predict it, even with class
weights (neutral weight was about 23). There are only 66 neutral tweets in train.

E9 tries **oversampling**: repeat every neutral training tweet 10 times (66 -> 660), so the model sees
neutral much more often during training. Same model (AfroXLMR, the best from 04), same settings,
same 3 seeds. Only the training data changes, so any difference comes from the oversampling.

Run on Colab with a T4 GPU (about 10 minutes).

In [ ]:
import os
REPO_URL = "https://github.com/Divineitu/pidgin-sentiment.git"
if not os.path.exists("../src") and not os.path.exists("pidgin-sentiment"):
    !git clone {REPO_URL}
if os.path.exists("pidgin-sentiment"):
    %cd pidgin-sentiment/notebooks
!pip install -q accelerate

In [ ]:
import os
import sys
import random
import shutil

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)
from sklearn.utils.class_weight import compute_class_weight

sys.path.append("../src")
from prepare_data import prepare
from preprocess import LABELS, LABEL2ID
from metrics import get_scores, print_report, plot_confusion, log_result

if not os.path.exists("../data/processed/train.csv"):
    prepare()

train = pd.read_csv("../data/processed/train.csv")
dev = pd.read_csv("../data/processed/dev.csv")
test_official = pd.read_csv("../data/processed/test_official.csv")
test_clean = pd.read_csv("../data/processed/test_clean.csv")
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Oversample neutral

In [ ]:
OVERSAMPLE = 10
neutral = train[train["label"] == "neutral"]
train_os = pd.concat([train] + [neutral] * (OVERSAMPLE - 1), ignore_index=True)
train_os = train_os.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print("before:", train["label"].value_counts().to_dict())
print("after: ", train_os["label"].value_counts().to_dict())

# class weights are computed on the new data, so neutral's weight drops (it's no longer that rare)
weights = compute_class_weight("balanced", classes=np.array(LABELS), y=train_os["label"])
class_weights = torch.tensor(weights, dtype=torch.float)
print(dict(zip(LABELS, weights.round(2))))

## Same training setup as notebook 04

In [ ]:
MODEL_ID = "Davlan/afro-xlmr-base"
SEEDS = [42, 1, 7]
MAX_LEN = 64
LR = 2e-5
EPOCHS = 5
BATCH = 32


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


def to_dataset(df, tokenizer):
    ds = Dataset.from_dict({"text": df["text"].tolist(), "labels": [LABEL2ID[l] for l in df["label"]]})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return get_scores([LABELS[i] for i in labels], [LABELS[i] for i in preds])


def predict(trainer, ds):
    logits = trainer.predict(ds).predictions
    return [LABELS[i] for i in logits.argmax(axis=-1)]


def fine_tune(train_df, seed, out_dir):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=out_dir, learning_rate=LR, per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=64, num_train_epochs=EPOCHS, weight_decay=0.01,
        warmup_steps=0.1,  # 10% of the training steps
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        fp16=torch.cuda.is_available(), seed=seed, report_to="none", logging_strategy="epoch",
    )
    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=to_dataset(train_df, tokenizer), eval_dataset=to_dataset(dev, tokenizer),
        data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer, tokenizer

## Run E9

In [ ]:
runs = []
best = {"dev_f1": -1}
splits = {"dev": dev, "test_official": test_official, "test_clean": test_clean}

for seed in SEEDS:
    print(f"===== E9 seed {seed} =====")
    out_dir = f"../models/tmp_E9_{seed}"
    trainer, tokenizer = fine_tune(train_os, seed, out_dir)
    for split, df in splits.items():
        pred = predict(trainer, to_dataset(df, tokenizer))
        scores = get_scores(df["label"], pred)
        runs.append({"exp_id": "E9", "seed": seed, "split": split, **scores})
        if split == "dev" and scores["macro_f1"] > best["dev_f1"]:
            best = {"dev_f1": scores["macro_f1"], "seed": seed}
            trainer.save_model("../models/e9_best")
            tokenizer.save_pretrained("../models/e9_best")
    del trainer
    shutil.rmtree(out_dir, ignore_errors=True)
    torch.cuda.empty_cache()

runs = pd.DataFrame(runs)
runs.to_csv("../results/e9_runs.csv", index=False)
print("best E9 run on dev:", best)

## Compare with E7 (same model, no oversampling)

In [ ]:
name = f"afro-xlmr-base + neutral oversampled x{OVERSAMPLE}"
means = runs.drop(columns=["exp_id", "seed"]).groupby("split").mean().round(4)
stds = runs.groupby("split")["macro_f1"].std().round(4)
for split in splits:
    log_result("E9", name, split, means.loc[split].to_dict(), f"mean of seeds {SEEDS}, macro_f1 std={stds[split]}")

log = pd.read_csv("../results/experiments.csv")
log[log["exp_id"].isin(["E7", "E9"])].sort_values(["split", "exp_id"])[
    ["exp_id", "split", "accuracy", "macro_f1", "f1_negative", "f1_neutral", "f1_positive"]]

In [ ]:
best_tok = AutoTokenizer.from_pretrained("../models/e9_best")
best_model = AutoModelForSequenceClassification.from_pretrained("../models/e9_best")
best_trainer = Trainer(model=best_model, data_collator=DataCollatorWithPadding(best_tok))
pred = predict(best_trainer, to_dataset(test_clean, best_tok))
print_report(test_clean["label"], pred)
plot_confusion(test_clean["label"], pred, "E9 oversampled neutral, clean test", "confusion_E9_test_clean.png")

out = test_clean.copy()
out["pred"] = pred
out.to_csv("../results/e9_predictions_test_clean.csv", index=False)

## Optional: if E9 is better, push it to the Hub for the app

Only run this if E9's macro-F1 beats E7 and neutral F1 is above 0.

In [ ]:
# from huggingface_hub import login
# login()
# best_model.push_to_hub("DivineItu/pidgin-sentiment")
# best_tok.push_to_hub("DivineItu/pidgin-sentiment")

In [ ]:
!zip -r e9_results.zip ../results/experiments.csv ../results/e9_runs.csv ../results/e9_predictions_test_clean.csv ../results/figures/confusion_E9_test_clean.png
from google.colab import files
files.download("e9_results.zip")